## Описание животного по тексту

In [1]:
pip install transformers accelerate

In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [3]:
from transformers import LlavaForConditionalGeneration, LlavaProcessor
import torch
from PIL import Image

In [4]:
model_id = "llava-hf/llava-1.5-7b-hf"

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

processor = LlavaProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [5]:
def extract_dog_attributes(image_path, model, processor, max_new_tokens=200):

    image = Image.open(image_path).convert("RGB")
    prompt = """USER: <image>
Describe dog's eyes color, fur texture, fur length, visible coat pattern or best estimate of coat color features (e.g. light paws, dark ears).
Also describe ears (shape and orientation).

Return concise structured description.
ASSISTANT
"""

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    result = processor.decode(output[0], skip_special_tokens=True)

    return result

In [8]:
image_path = "dog_path.jpg"
result = extract_dog_attributes(
    image_path,
    model,
    processor
)

print(result)

USER:  
Describe dog's eyes color, fur texture, fur length, visible coat pattern or best estimate of coat color features (e.g. light paws, dark ears).
Also describe ears (shape and orientation).

Return concise structured description.
ASSISTANT
The dog has brown eyes, a white fur coat, and a blue collar. Its fur is short and smooth, and it appears to be a large white dog. The dog's ears are perked up, and it is standing in a forest with yellow leaves around it.


## 2. Добавление особенностей постов приюта.

Объединим все наши данные в одну таблицу.

In [9]:
import pandas as pd

In [10]:
df = pd.concat(
    [pd.read_csv("bir.csv"),
    pd.read_csv("final_yuna.csv"),
    pd.read_csv("iskra.csv"),
    pd.read_csv("nek.csv"),
    pd.read_csv("pp_animals.csv")], axis=0, ignore_index=True)
df

,Unnamed: 0,animal_id,media_id,shelter_id,species,animal_name,source_url,description_short,detail_name,description_raw,attributes_raw_text,attributes_raw_json
0,0,1,animal_card,4,dog,Фета,https://izpriuta.ru/sobaki/feta,Манеры и воспитание принцессы :),Фета,"Малышке Фете всего 3,5 месяца. Это невероятно ...","Малышке Фете всего 3,5 месяца. Это невероятно ...","{""full_text"": ""Малышке Фете всего 3,5 месяца. ..."
1,1,2,animal_card,4,dog,Рокфор,https://izpriuta.ru/sobaki/rokfor,Юный и преданный друг!,Рокфор,"Знакомьтесь, это чудо зовут Рокфор. Ему всего ...","Знакомьтесь, это чудо зовут Рокфор. Ему всего ...","{""full_text"": ""Знакомьтесь, это чудо зовут Рок..."
2,2,3,animal_card,4,dog,Чеддер,https://izpriuta.ru/sobaki/chedder,В мечтах о счастливом детстве,Чеддер,"Чеддер — совсем кроха, ему около 3,5 месяцев, ...","Чеддер — совсем кроха, ему около 3,5 месяцев, ...","{""full_text"": ""Чеддер — совсем кроха, ему окол..."
3,3,4,animal_card,4,dog,Удо,https://izpriuta.ru/sobaki/udo,Способный парень,Удо,"Он настоящее чудо и живая эмоция, Удо из тех, ...","Он настоящее чудо и живая эмоция, Удо из тех, ...","{""full_text"": ""Он настоящее чудо и живая эмоци..."
4,4,5,animal_card,4,dog,Лекси,https://izpriuta.ru/sobaki/leksi,"Леди, как с обложки модного журнала",Лекси,"Если бы Лекси попала в индустрию моды, то точн...","Если бы Лекси попала в индустрию моды, то точн...","{""full_text"": ""Если бы Лекси попала в индустри..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2557,568,33,animal_card,2,cat,Чарли,https://cats.pechatniki-pets.ru/charli,NaN,NaN,❤️ Общительный котик с красивой чёрной шёрстко...,❤️ Общительный котик с красивой чёрной шёрстко...,"{""full_text"": ""❤️ Общительный котик с красивой..."
2558,569,34,animal_card,2,cat,Чинара,https://cats.pechatniki-pets.ru/chinara,NaN,NaN,❤️ Красивая крупная кошка из приюта [Печатники...,❤️ Красивая крупная кошка из приюта [Печатники...,"{""full_text"": ""❤️ Красивая крупная кошка из пр..."
2559,570,35,animal_card,2,cat,Штрудель,https://cats.pechatniki-pets.ru/shtrudel,NaN,NaN,❤️ Красивый пушистый кот серо-белого окраса жи...,❤️ Красивый пушистый кот серо-белого окраса жи...,"{""full_text"": ""❤️ Красивый пушистый кот серо-б..."
2560,571,36,animal_card,2,cat,Эркюль,https://cats.pechatniki-pets.ru/erkul,NaN,NaN,❤️ Замечательный кот живет в московском приюте...,❤️ Замечательный кот живет в московском приюте...,"{""full_text"": ""❤️ Замечательный кот живет в мо..."


In [11]:
pip install sentence-transformers

In [14]:
df["attributes_raw_text"]

,attributes_raw_text
0,"Малышке Фете всего 3,5 месяца. Это невероятно ..."
1,"Знакомьтесь, это чудо зовут Рокфор. Ему всего ..."
2,"Чеддер — совсем кроха, ему около 3,5 месяцев, ..."
3,"Он настоящее чудо и живая эмоция, Удо из тех, ..."
4,"Если бы Лекси попала в индустрию моды, то точн..."
...,...
2557,❤️ Общительный котик с красивой чёрной шёрстко...
2558,❤️ Красивая крупная кошка из приюта [Печатники...
2559,❤️ Красивый пушистый кот серо-белого окраса жи...
2560,❤️ Замечательный кот живет в московском приюте...


In [15]:
df["attributes_raw_text"].apply(type).value_counts()

,count
attributes_raw_text,
<class 'str'>,2242
<class 'float'>,320


Заполняем все пропуски нулевыми строчками, чтобы они все имели одинаковый тип.

In [16]:
df["attributes_raw_text"] = df["attributes_raw_text"].fillna("").astype(str)

Здесь мы отбираем посты того же приюта.

In [ ]:
def retrieve_similar_posts(
    df,
    shelter_id: int,
):
    subset = df[df["shelter_id"] == shelter_id].copy()
    return subset

In [ ]:
def analyze_shelter_style(retrieved_texts, llm):
    texts_block = "\n\n---\n\n".join(retrieved_texts)

    prompt = f"""
        Ты — аналитик текстов приюта животных.
        Твоя задача — проанализировать примеры постов приюта и описать их стиль письма так, чтобы другой текст можно было написать в точно такой же манере.
        Проанализируй тексты и выдели только наблюдаемые характеристики стиля.
        Не придумывай правил — опирайся только на данные.

        Верни описание стиля в JSON:

        {{
          "tone": "какая общая тональность текста (например: тёплая, эмоциональная, нейтральная и т.д.)",
          "emoji_usage": "есть ли эмодзи, как часто используются, какие роли выполняют",
          "lexicon_features": "типичные слова/выражения/уменьшительные формы/обращения",
          "structure_patterns": "как обычно устроены тексты (если есть закономерности)",
          "typical_length": "примерная длина постов (короткие/средние/длинные или диапазон)",
          "communication_style": "как автор обращается к читателю и описывает животных",
          "overall_summary": "краткое резюме стиля в 1–2 предложениях"
        }}

        Тексты приюта:
        {texts_block}
        """

    return llm(prompt)

Эти данные вводит волонтер в бот.

In [ ]:
name = "glafira"
age = 18
gender = "female"
traits = "добри послушни"

In [ ]:
def generate_final_post(
    photo_description: str,
    shelter_style: str,
    shelter_id: int,
    llm
):
    prompt = f"""
Ты — копирайтер приюта животных.

Твоя задача — написать пост о собаке, строго следуя стилю приюта.
СТИЛЬ ПРИЮТА (из таких же постов):
{shelter_style}

ИНФОРМАЦИЯ О СОБАКЕ:
- имя: {name}
- возраст: {age}
- пол: {gender}
- описание внешности (из изображения): {photo_description}
- характер (если есть): {traits}

ПРАВИЛА:
- используй стиль приюта максимально точно (тон, лексику, длину, эмодзи если они характерны)
- длина текста должна быть похожа на типичные посты приюта
- не добавляй новых фактов о собаке
- не выдумывай историю, болезни или поведение
- не навязывай структуру — следуй естественному стилю приюта
- если в стиле приюта нет эмодзи — не используй их

ЗАДАЧА:
Сгенерировать текст, который выглядит так, будто его написал такой же волонтер как и другие тексты этого приюта в том же стиле.

Верни только готовый текст поста.
"""

    return llm(prompt)

In [ ]:
def rag_generate_post(
    df,
    photo,
    shelter_id: int,
    llm,
    top_k: int = 100
):
    # 1. описание по фото
    photo_description = extract_dog_attributes(image_path, model, processor)

    # 2. retrieval
    retrieved_texts = retrieve_similar_posts(
        df=df,
        shelter_id=shelter_id,
        query_text=photo_description,
        top_k=top_k
    )

    # 3. анализ стиля
    shelter_style = analyze_shelter_style(
        retrieved_texts=retrieved_texts,
        llm=llm
    )

    # 4. финальный пост
    final_post = generate_final_post(
        photo_description=photo_description,
        shelter_style=shelter_style,
        shelter_id=shelter_id,
        llm=llm
    )

    return final_post

Тут используем гигачат


In [101]:
!pip install gigachat

In [ ]:

import base64
from gigachat import GigaChat


raw = "tut_api"
credentials = base64.b64encode(raw.encode()).decode()

llm_client = GigaChat(
    credentials=credentials,
    verify_ssl_certs=False
)

SYSTEM_PROMPT = "Ты помощник, который пишет трогательные и честные тексты для приютов животных."

def call_llm(prompt: str) -> str:
    full_prompt = f"{SYSTEM_PROMPT}\n\nПользователь: {prompt}"

    response = llm_client.chat(full_prompt)

    try:
        return response.choices[0].message.content
    except:
        return response.choices[0].text

In [ ]:
post_text = rag_generate_post(
    df=df,
    photo="dog_sample.jpg",
    shelter_id=5,
    llm=call_llm
)

print(post_text)